# Q1: Expected home and away corners

**Goal.** For each betting match, output expected home corners μ_h and expected
away corners μ_a (plus per-side conditional variance σ²). These feed Q2.

**Architecture — three stages.**

1. **Stage 1a — Team-strength model.** Sequential Bayesian Poisson updater
   (Elo-style) that maintains per-team latent (att, def) in log-corner space.
   Walks all matches with observed corners (corners_data + betting outcomes)
   chronologically; each match's features are read *before* its update, so
   they are leak-free by construction. A second pass with venue-specific
   latents gives at-home / at-away strengths. Output: 10–12 dense features
   per match capturing "how strong is this team, how strong is its opponent".
2. **Stage 1b — Variance & form features.** Per-team rolling std (corner
   variance), short EWM (recent form), goal-based rolling (an independent
   view of attacking strength), and form-residual = recent form minus what
   Stage 1a's strength predicts.
3. **Stage 2 — Main GBT.** LightGBM + XGBoost blend (L2 loss) consuming
   Stage 1a + Stage 1b features + match context (competition, season,
   gameweek) + 1X2-implied probabilities (only available on betting set;
   NaN-tolerant). Predicts μ_h, μ_a, μ_total separately.

**Stage 3 — Dispersion head + clean Q2 holdout.** The 1911 betting matches
are split chronologically: A = first 40% (calibration), B = remaining 60%
(Q2 holdout). On A, fit log σ² as a function of μ, |p_h_1x2 − 0.5|, and
team-variance features. μ is *not* adjusted on A so Q2's edge is
preserved — the dispersion only sharpens NB probabilities for EV.

**Why this works.** Stage 1a captures team identity (e.g. team 698 mean
3.6 vs league 5.4) as 1–2 numbers per side instead of letting the GBT
hunt for it across many EWM features. The GBT then spends its capacity
on matchup interactions and high-variance situations rather than
re-deriving "this is a low-corner team" via deep splits. 1X2 odds enter
only as features on the betting set and as a variance signal in Stage 3,
preserving Q2 EV edge against the OU/HC prices.

**Pipeline cells.**
1. Load corners_data, betting, train/val pools
2. Baseline expectations (Poisson floor + mean baseline)
3. **Stage 1a — team strength features** (general + venue-split)
4. Reshape to team-game long format
5. Stage 1b rolling features (EWM, goals, variance)
6. Pivot back + matchup interactions
7. Build betting-set features + 1X2 implied probs
8. NaN audit / fill
9. Chronological train/val split
10. Train LightGBM + XGBoost on home / away / total
11. Evaluate on val (MAE / RMSE / NLL / CRPS) + calibration
12. Stage 3 — split betting A/B, fit dispersion head on A
13. Save predictions for Q2 (with partition + σ² columns)

## Load data and restrict to betting competitions

`train.parquet` is the modelling pool (historical matches with corner outcomes
and the importance weight `iw` to reweight to the betting distribution).
`betting.parquet` is the holdout: matches with prices that Q2 will evaluate.
We restrict the training pool to competitions that appear in the betting set,
so per-competition rolling and league baselines are not diluted by leagues
we will never predict on.

In [1]:
import numpy as np
import pandas as pd
from scipy.stats import poisson
from sklearn.metrics import mean_absolute_error, mean_squared_error
from lightgbm import LGBMRegressor, early_stopping
from xgboost import XGBRegressor

train = pd.read_parquet('train.parquet')
betting = pd.read_parquet('betting.parquet')
train['date_time']   = pd.to_datetime(train['date_time'])
betting['date_time'] = pd.to_datetime(betting['date_time'])

betting_matches = betting.drop_duplicates('match_id').copy()
model_data = train[train['competition_id'].isin(betting_matches['competition_id'])].copy()
model_data = model_data.sort_values('date_time').reset_index(drop=True)
model_data['total_corners'] = model_data['home_corners'] + model_data['away_corners']

print('model_data:', model_data.shape, '| betting_matches:', betting_matches.shape)
print('train date range:',  model_data['date_time'].min().date(), 'to', model_data['date_time'].max().date())
print('betting date range:', betting_matches['date_time'].min().date(), 'to', betting_matches['date_time'].max().date())

model_data: (11198, 14) | betting_matches: (1911, 17)
train date range: 2020-08-08 to 2024-12-08
betting date range: 2024-08-09 to 2025-05-11


## Baseline expectations: mean baseline + Poisson irreducible MAE

Two reference points before any modelling:

**Mean baseline.** Always predict the training mean. Any useful model must beat
this; how much it beats it tells us how much *team-specific* signal the features
carry beyond the global average.

**Poisson irreducible MAE = √(2λ/π).** Even a model that knows the *exact* true
rate λ for each match will still average |y − λ| ≈ √(2λ/π) of error because the
realised count y is a Poisson draw. This is the noise floor — we cannot go below
it without leakage. The gap between mean baseline and floor is the *improvable
range*; what fraction of it our model captures is the real measure of skill.

In [2]:
targets = ['home_corners', 'away_corners', 'total_corners']
disp = pd.DataFrame({'mean': model_data[targets].mean(),
                     'variance': model_data[targets].var()})
disp['variance / mean'] = disp['variance'] / disp['mean']
display(disp.round(3))

np.random.seed(0)
def irreducible_mae(lam, n=200_000):
    return float(np.mean(np.abs(np.random.poisson(lam, n) - lam)))
bound = pd.DataFrame([
    {'target': t,
     'lambda': disp.loc[t, 'mean'],
     'sqrt(2*lambda/pi)': float(np.sqrt(2 * disp.loc[t, 'mean'] / np.pi)),
     'simulated_MAE':     irreducible_mae(disp.loc[t, 'mean'])}
    for t in targets
])
display(bound.round(3))

,mean,variance,variance / mean
home_corners,5.393,8.519,1.580
away_corners,4.445,6.792,1.528
total_corners,9.839,11.373,1.156


,target,lambda,sqrt(2*lambda/pi),simulated_MAE
0,home_corners,5.393,1.853,1.864
1,away_corners,4.445,1.682,1.696
2,total_corners,9.839,2.503,2.494


## Stage 1a: team-strength model (sequential Bayesian Poisson)

For each team t, maintain latent (att_t, def_t) in log-corner space —
zero means league-average, positive att means stronger attack, positive
def means leakier defence (concedes more corners). The match's expected
corners are:

```
log λ_h = log(comp_baseline_home) + att_h + def_a
log λ_a = log(comp_baseline_away) + att_a + def_h
```

Walk every match with observed corners chronologically. Before applying
each match's update, read out the current (att, def) for both teams as
features for *that* match — leak-free by construction. After the
match, update the four latents (att_h, def_a, att_a, def_h) by their
log-residuals with a learning rate `η_t = max_eta · K / (n_obs_t + K)`
that decays with experience (K=15 matches gives 50% pull at 15 prior
games, matching the shrinkage strength used elsewhere).

Two passes:

- **General**: each team has one (att, def). Updated from every match.
  Captures overall strength.
- **Venue-split**: each team has separate (att_home, def_home,
  att_away, def_away). Each pair updated only from matches where the
  team played that venue. Captures home/away tendencies that go beyond
  the per-(competition, season) home advantage already in the baseline.

Update pool = `corners_data` (11k historical matches) plus betting
outcomes (1.9k more). Targets = train.parquet matches plus
betting_matches. Stage 1a features are then merged into the main
feature pipeline.

In [3]:
from team_strength import walk_matches

all_matches = pd.read_parquet('all_matches.parquet')
all_matches['date_time'] = pd.to_datetime(all_matches['date_time'])

target_match_ids = set(model_data['match_id'].astype(int)) | set(betting_matches['match_id'].astype(int))
strength_features = walk_matches(all_matches, target_match_ids)

print(f'Stage 1a strength features: {strength_features.shape}')
print(f'update pool: {len(all_matches):,} matches  |  target pool: {len(target_match_ids):,} matches')
print(f'home_attack_rating std={strength_features.home_attack_rating.std():.3f}  '
      f'range=[{strength_features.home_attack_rating.min():.2f},{strength_features.home_attack_rating.max():.2f}]')
print(f'strength_diff std={strength_features.strength_diff.std():.3f}  '
      f'venue_strength_diff std={strength_features.venue_strength_diff.std():.3f}')

Stage 1a strength features: (13109, 17)
update pool: 13,484 matches  |  target pool: 13,109 matches
home_attack_rating std=0.099  range=[-0.35,0.28]
strength_diff std=0.263  venue_strength_diff std=0.172


## Team-game long format

**The problem.** A match-level row has two teams (home and away). To compute
*team form* — e.g. "team X's average corners over their last 10 matches" — we
need each team's chronological history. From a match-level dataframe, this would
require two separate per-team computations (one for the home team, one for the
away), then a join, which is awkward and bug-prone.

**The trick.** Stack the dataframe so each match becomes *two* rows, one per team:
columns become `team_id`, `corners_for`, `corners_against`, `is_home`. Now
`groupby('team_id')` gives every team's chronological game history naturally,
and any rolling / EWM / shift transform applies trivially. After computing
features, pivot back to match level: take the `is_home == 1` rows as the home
team's features and `is_home == 0` rows as the away team's.

**Why `shift(1)` matters.** Every rolling feature is computed as
`s.shift(1).rolling(w).mean()` so the feature for match M uses only matches
*strictly before* M. Without `shift(1)` the model would see its own target
in the rolling window — pure leakage.

In [4]:
home_rows = model_data[['match_id', 'date_time', 'competition_id',
                        'home_team_id', 'home_corners', 'away_corners',
                        'home_ft_score', 'away_ft_score']].rename(
    columns={'home_team_id': 'team_id',
             'home_corners': 'corners_for', 'away_corners': 'corners_against',
             'home_ft_score': 'goals_for',  'away_ft_score': 'goals_against'})
home_rows['is_home'] = 1

away_rows = model_data[['match_id', 'date_time', 'competition_id',
                        'away_team_id', 'away_corners', 'home_corners',
                        'away_ft_score', 'home_ft_score']].rename(
    columns={'away_team_id': 'team_id',
             'away_corners': 'corners_for', 'home_corners': 'corners_against',
             'away_ft_score': 'goals_for',  'home_ft_score': 'goals_against'})
away_rows['is_home'] = 0

team_games = pd.concat([home_rows, away_rows], ignore_index=True)
team_games = team_games.sort_values(['team_id', 'date_time', 'match_id']).reset_index(drop=True)

print(f'match-level rows: {len(model_data):,}  ->  team-game rows: {len(team_games):,}  (= 2 x match rows)')
display(team_games.head(4))

match-level rows: 11,198  ->  team-game rows: 22,396  (= 2 x match rows)


,match_id,date_time,competition_id,team_id,corners_for,corners_against,goals_for,goals_against,is_home
0,6632841,2020-09-13 19:00:00,795,674,3,4,4,2,1
1,6632836,2020-09-19 19:00:00,795,674,6,6,1,2,0
2,6632821,2020-09-26 14:00:00,795,674,4,4,1,1,1
3,6632811,2020-09-29 17:00:00,795,674,3,9,1,0,0


## Stage 1b: rolling form, goals, and team variance

Stage 1a already captures stable team strength and matchup expectation.
Stage 1b adds three families that complement it:

- **EWM short form (`hl=5`)** — recent corner-for and corner-against,
  shifted by 1. Captures form swings (injury wave, new manager) that
  Stage 1a's slow-updating latents lag on.
- **Goal rolling (`l20`)** — `gf_l20`, `ga_l20`, `gd_l20`. Goals are an
  *independent* signal of attacking strength: a strong attack creates
  corners *and* goals from the same underlying pressure, but corner
  noise and goal noise are roughly uncorrelated. Adding goal rolling
  gives the GBT a second, lower-noise view of team strength.
- **Variance rolling (`l20`)** — per-team rolling std of corners-for
  and corners-against. Lets the GBT see "this team has high week-to-week
  variance" and split harder on opponent / competition for those teams.

Drop the old short-window venue, opponent-adjusted, and `hl=10` EWM
families: they're redundant given Stage 1a's venue-split latents and
already-shrunk strength estimates. Pruning these reduces feature noise.

In [5]:
team_games['cf_ewm_hl5'] = team_games.groupby('team_id')['corners_for'].transform(
    lambda s: s.shift(1).ewm(halflife=5, ignore_na=True).mean())
team_games['ca_ewm_hl5'] = team_games.groupby('team_id')['corners_against'].transform(
    lambda s: s.shift(1).ewm(halflife=5, ignore_na=True).mean())

team_games['gf_l20'] = team_games.groupby('team_id')['goals_for'].transform(
    lambda s: s.shift(1).rolling(20, min_periods=3).mean())
team_games['ga_l20'] = team_games.groupby('team_id')['goals_against'].transform(
    lambda s: s.shift(1).rolling(20, min_periods=3).mean())
team_games['gd_l20'] = team_games['gf_l20'] - team_games['ga_l20']

team_games['cf_std_l20'] = team_games.groupby('team_id')['corners_for'].transform(
    lambda s: s.shift(1).rolling(20, min_periods=5).std())
team_games['ca_std_l20'] = team_games.groupby('team_id')['corners_against'].transform(
    lambda s: s.shift(1).rolling(20, min_periods=5).std())

team_games['n_prior'] = team_games.groupby('team_id').cumcount()

print('team_games with rolling features:', team_games.shape)
print('NaN per new feature:')
display(team_games[['cf_ewm_hl5', 'ca_ewm_hl5', 'gf_l20', 'ga_l20',
                    'cf_std_l20', 'ca_std_l20']].isna().sum())

team_games with rolling features: (22396, 17)
NaN per new feature:


cf_ewm_hl5    180
ca_ewm_hl5    180
gf_l20        540
ga_l20        540
cf_std_l20    900
ca_std_l20    900
dtype: int64

## Pivot back to match level + Stage 1a merge + matchup interactions

Take `is_home == 1` team-game rows for the home team's Stage 1b features
(EWM, goals, variance), `is_home == 0` for the away team's, and merge
both onto `model_data` by `match_id`. Then merge in Stage 1a strength
features (already at match level: home_att, away_att, …,
home_att_at_home, away_att_at_away, …) and build matchup interactions:

- `form_resid_h` = `home_cf_ewm_hl5 − exp(home_att + comp_log_lam_h)` —
  recent form *minus* what Stage 1a's strength predicts. Positive
  means "scoring corners above their level" (improving / hot streak).
- `form_resid_a` = symmetric for the away side.
- `gd_diff` = `home_gd_l20 − away_gd_l20` — proxy for 1X2 win-prob
  differential built from goals.
- `attack_vs_defence_h` = `home_att + away_def` — additive matchup in
  log space; large positive means strong attack vs leaky defence.
- `attack_vs_defence_a` = symmetric.

In [6]:
team_feature_cols = ['cf_ewm_hl5', 'ca_ewm_hl5',
                     'gf_l20', 'ga_l20', 'gd_l20',
                     'cf_std_l20', 'ca_std_l20']

home_feats = (team_games[team_games['is_home'] == 1][['match_id'] + team_feature_cols]
              .rename(columns={c: f'home_{c}' for c in team_feature_cols}))
away_feats = (team_games[team_games['is_home'] == 0][['match_id'] + team_feature_cols]
              .rename(columns={c: f'away_{c}' for c in team_feature_cols}))

features = (model_data
            .merge(home_feats, on='match_id', how='left')
            .merge(away_feats, on='match_id', how='left')
            .merge(strength_features, on='match_id', how='left'))

features['form_resid_h'] = features['home_cf_ewm_hl5'] - np.exp(features['home_attack_rating'] + features['league_log_baseline_home'])
features['form_resid_a'] = features['away_cf_ewm_hl5'] - np.exp(features['away_attack_rating'] + features['league_log_baseline_away'])
features['gd_diff'] = features['home_gd_l20'] - features['away_gd_l20']
features['attack_vs_defence_h'] = features['home_attack_rating'] + features['away_defence_leakiness']
features['attack_vs_defence_a'] = features['away_attack_rating'] + features['home_defence_leakiness']
features['attack_vs_defence_h_venue'] = features['home_attack_at_home'] + features['away_defence_at_away']
features['attack_vs_defence_a_venue'] = features['away_attack_at_away'] + features['home_defence_at_home']

print('features:', features.shape)

features: (11198, 51)


## Build betting-set features

For each betting match, take the latest team-level Stage 1b snapshot
strictly before kickoff via `merge_asof(direction='backward')` for
EWM / goal / variance features. Stage 1a strength features were already
computed for betting matches (point-in-time inside Stage 1a) and merge
in by `match_id`. Finally, attach 1X2 implied probabilities (de-vigged):

```
p_h = (1/oh) / (1/oh + 1/oa + 1/od)
```

1X2 odds appear in `betting.parquet` with `odds_type == '1X2'` for 1906
of 1911 matches; the missing 5 are filled with the no-information prior
(1/3, 1/3, 1/3). Train matches do not have 1X2 prices — those features
are NaN there. LightGBM and XGBoost handle missing natively, learning a
"no 1X2 available" default split for the train period, then leveraging
the actual probabilities on the betting set.

In [7]:
team_snap = (team_games[['team_id', 'date_time'] + team_feature_cols]
             .sort_values(['date_time', 'team_id']).reset_index(drop=True))

def latest(fixtures, side, snap, cols):
    col = f'{side}_team_id'
    out = fixtures[['match_id', 'date_time', col]].rename(columns={col: 'team_id'}).copy()
    out = out.sort_values('date_time').reset_index(drop=True)
    out = pd.merge_asof(out, snap, on='date_time', by='team_id', direction='backward')
    return out[['match_id'] + cols].rename(columns={c: f'{side}_{c}' for c in cols})

bet_h = latest(betting_matches, 'home', team_snap, team_feature_cols)
bet_a = latest(betting_matches, 'away', team_snap, team_feature_cols)

x12 = (betting[betting['odds_type'] == '1X2']
       .drop_duplicates('match_id')[['match_id', 'oh', 'oa', 'od']].copy())
inv_sum = 1 / x12['oh'] + 1 / x12['oa'] + 1 / x12['od']
x12['p_h_1x2'] = (1 / x12['oh']) / inv_sum
x12['p_a_1x2'] = (1 / x12['oa']) / inv_sum
x12['p_d_1x2'] = (1 / x12['od']) / inv_sum
x12 = x12[['match_id', 'p_h_1x2', 'p_a_1x2', 'p_d_1x2']]

betting_match_features = (betting_matches
                          .merge(bet_h, on='match_id', how='left')
                          .merge(bet_a, on='match_id', how='left')
                          .merge(strength_features, on='match_id', how='left')
                          .merge(x12, on='match_id', how='left')
                          .sort_values(['date_time', 'competition_id'])
                          .reset_index(drop=True))

for c in ['p_h_1x2', 'p_a_1x2', 'p_d_1x2']:
    betting_match_features[c] = betting_match_features[c].fillna(1/3)

betting_match_features['form_resid_h'] = (betting_match_features['home_cf_ewm_hl5']
    - np.exp(betting_match_features['home_attack_rating'] + betting_match_features['league_log_baseline_home']))
betting_match_features['form_resid_a'] = (betting_match_features['away_cf_ewm_hl5']
    - np.exp(betting_match_features['away_attack_rating'] + betting_match_features['league_log_baseline_away']))
betting_match_features['gd_diff'] = betting_match_features['home_gd_l20'] - betting_match_features['away_gd_l20']
betting_match_features['attack_vs_defence_h'] = betting_match_features['home_attack_rating'] + betting_match_features['away_defence_leakiness']
betting_match_features['attack_vs_defence_a'] = betting_match_features['away_attack_rating'] + betting_match_features['home_defence_leakiness']
betting_match_features['attack_vs_defence_h_venue'] = betting_match_features['home_attack_at_home'] + betting_match_features['away_defence_at_away']
betting_match_features['attack_vs_defence_a_venue'] = betting_match_features['away_attack_at_away'] + betting_match_features['home_defence_at_home']

# train.parquet matches have no 1X2 -- add NaN columns to features so train/val/bet share schema
for c in ['p_h_1x2', 'p_a_1x2', 'p_d_1x2']:
    features[c] = np.nan

print('betting_match_features:', betting_match_features.shape)
print('1X2 coverage:', betting_match_features[['p_h_1x2','p_a_1x2','p_d_1x2']].notna().all(axis=1).mean())

betting_match_features: (1911, 57)
1X2 coverage: 1.0


## NaN audit

Three structural sources of NaN in the new feature set:

1. **Team rolling (Stage 1b)**: `cf_ewm_hl5`, `gf_l20`, `cf_std_l20` etc.
   are NaN for each team's *first ever match* in the dataset
   (`shift(1)` plus `min_periods` thresholds).
2. **Stage 1a**: `home_att`, `home_def` etc. start at 0.0 (= league
   average) for every team's first match by construction, so these are
   never NaN. `comp_log_lam_h/a` falls back to the global league mean
   when a (competition, season) has no prior match.
3. **`merge_asof` on betting**: NaN if a team has no prior match in
   `team_snap` before the betting kickoff. Rare.
4. **1X2 on train**: train.parquet has no odds, so `p_*_1x2` is NaN for
   all train rows. Boosters handle this natively; we leave it as NaN.

In [8]:
meta_cols = {'match_id', 'date_time', 'competition_id', 'season_id', 'gameweek',
             'iw', 'home_team_id', 'away_team_id',
             'home_corners', 'away_corners', 'total_corners',
             'home_ft_score', 'away_ft_score', 'corner_diff_home_minus_away',
             'odds_type', 'oh', 'oa', 'od'}

train_feat_cols = [c for c in features.columns                 if c not in meta_cols]
bet_feat_cols   = [c for c in betting_match_features.columns   if c not in meta_cols]

nan_train = features[train_feat_cols].isna().sum()
nan_bet   = betting_match_features[bet_feat_cols].isna().sum()

print(f'features:                NaN cells = {nan_train.sum():,}  ({nan_train.sum() / (len(features) * len(train_feat_cols)):.2%})')
print(f'betting_match_features:  NaN cells = {nan_bet.sum():,}  ({nan_bet.sum() / (len(betting_match_features) * len(bet_feat_cols)):.2%})')
print('\ntop columns with NaN (features):')
display(nan_train[nan_train > 0].sort_values(ascending=False).head(15))
print('\ntop columns with NaN (betting_match_features):')
display(nan_bet[nan_bet > 0].sort_values(ascending=False).head(15))

features:                NaN cells = 37,889  (8.46%)
betting_match_features:  NaN cells = 1,581  (2.07%)

top columns with NaN (features):


p_h_1x2            11198
p_a_1x2            11198
p_d_1x2            11198
away_cf_std_l20      452
away_ca_std_l20      452
home_cf_std_l20      448
home_ca_std_l20      448
gd_diff              335
home_gd_l20          271
home_ga_l20          271
home_gf_l20          271
away_gf_l20          269
away_ga_l20          269
away_gd_l20          269
away_ca_ewm_hl5       91
dtype: int64


top columns with NaN (betting_match_features):


gd_diff            173
home_cf_ewm_hl5     89
home_ca_ewm_hl5     89
home_ga_l20         89
home_gf_l20         89
home_cf_std_l20     89
home_ca_std_l20     89
form_resid_h        89
home_gd_l20         89
away_ca_ewm_hl5     87
away_cf_ewm_hl5     87
away_gf_l20         87
away_ga_l20         87
away_cf_std_l20     87
away_gd_l20         87
dtype: int64

## NaN fill

Fill order:
- Stage 1b rolling (EWM, goals, variance) → league mean / league std for
  the first-match-ever rows where rolling could not be computed.
- `comp_log_lam_h/a` → log of global mean (already filled inside Stage
  1a, but defensive fill here for safety).
- Stage 1a outputs (`home_att` … `away_def_at_away`) → 0 (= league
  average), the prior they're initialised to.
- Derived (`form_resid_h/a`, `gd_diff`, `attack_vs_defence_*`) → recompute
  after the underlying fills.
- `p_*_1x2` on train → leave NaN (booster handles natively).

In [9]:
global_mean_h = model_data['home_corners'].mean()
global_mean_a = model_data['away_corners'].mean()
global_mean_g = model_data[['home_ft_score', 'away_ft_score']].mean().mean()
global_std_c  = float(np.sqrt(0.5 * (model_data['home_corners'].var() + model_data['away_corners'].var())))
log_mean_h    = float(np.log(global_mean_h))
log_mean_a    = float(np.log(global_mean_a))

stage1a_cols = ['home_attack_rating', 'home_defence_leakiness',
                'away_attack_rating', 'away_defence_leakiness',
                'log_lambda_home', 'log_lambda_away', 'strength_diff',
                'home_attack_at_home', 'home_defence_at_home',
                'away_attack_at_away', 'away_defence_at_away',
                'venue_strength_diff']

for df, mean_h, mean_a in [(features, global_mean_h, global_mean_a),
                           (betting_match_features, global_mean_h, global_mean_a)]:
    df['home_cf_ewm_hl5'] = df['home_cf_ewm_hl5'].fillna(mean_h)
    df['home_ca_ewm_hl5'] = df['home_ca_ewm_hl5'].fillna(mean_a)
    df['away_cf_ewm_hl5'] = df['away_cf_ewm_hl5'].fillna(mean_a)
    df['away_ca_ewm_hl5'] = df['away_ca_ewm_hl5'].fillna(mean_h)
    for c in ['home_gf_l20', 'home_ga_l20', 'away_gf_l20', 'away_ga_l20']:
        df[c] = df[c].fillna(global_mean_g)
    df['home_gd_l20'] = df['home_gf_l20'] - df['home_ga_l20']
    df['away_gd_l20'] = df['away_gf_l20'] - df['away_ga_l20']
    for c in ['home_cf_std_l20', 'home_ca_std_l20', 'away_cf_std_l20', 'away_ca_std_l20']:
        df[c] = df[c].fillna(global_std_c)

    df['league_log_baseline_home'] = df['league_log_baseline_home'].fillna(log_mean_h)
    df['league_log_baseline_away'] = df['league_log_baseline_away'].fillna(log_mean_a)
    for c in stage1a_cols:
        df[c] = df[c].fillna(0.0)
    if 'prior_match_count_home' in df.columns:
        df['prior_match_count_home'] = df['prior_match_count_home'].fillna(0)
        df['prior_match_count_away'] = df['prior_match_count_away'].fillna(0)

    df['form_resid_h'] = df['home_cf_ewm_hl5'] - np.exp(df['home_attack_rating'] + df['league_log_baseline_home'])
    df['form_resid_a'] = df['away_cf_ewm_hl5'] - np.exp(df['away_attack_rating'] + df['league_log_baseline_away'])
    df['gd_diff'] = df['home_gd_l20'] - df['away_gd_l20']
    df['attack_vs_defence_h'] = df['home_attack_rating'] + df['away_defence_leakiness']
    df['attack_vs_defence_a'] = df['away_attack_rating'] + df['home_defence_leakiness']
    df['attack_vs_defence_h_venue'] = df['home_attack_at_home'] + df['away_defence_at_away']
    df['attack_vs_defence_a_venue'] = df['away_attack_at_away'] + df['home_defence_at_home']

print('NaN after fill -- features (excl. p_*_1x2 on train):')
print(features[[c for c in train_feat_cols if not c.endswith('_1x2')]].isna().sum().sum())
print('NaN after fill -- betting_match_features:')
print(betting_match_features[bet_feat_cols].isna().sum().sum())

NaN after fill -- features (excl. p_*_1x2 on train):
0
NaN after fill -- betting_match_features:
0


## Train / validation / test split

Three sets, all chronological — never random:

- **train** (80% earliest of `train.parquet`): used to fit the boosting trees.
- **val**   (20% latest of `train.parquet`):   used for early stopping and model
  comparison. Reported MAE here is mildly optimistic since early-stopping picks
  the best iteration on this set.
- **test / holdout** (`betting.parquet`):       the betting universe — disjoint
  from train.parquet by construction. We predict μ_h, μ_a, μ_t on this set and
  hand off to Q2 for probability + EV. Q2 will use the realised corners stored
  on this set as ground truth.

A random 80/20 split would leak future matches into training (e.g. some January
2024 matches in train, others in val); chronological avoids that. The same
principle applies to feature construction: every rolling / expanding window uses
`shift(1)` so a match never sees its own outcome.

In [10]:
feature_cols = [
    'competition_id', 'season_id', 'gameweek',
    'home_attack_rating', 'home_defence_leakiness',
    'away_attack_rating', 'away_defence_leakiness',
    'home_attack_at_home', 'home_defence_at_home',
    'away_attack_at_away', 'away_defence_at_away',
    'log_lambda_home', 'log_lambda_away',
    'strength_diff', 'venue_strength_diff',
    'attack_vs_defence_h', 'attack_vs_defence_a',
    'attack_vs_defence_h_venue', 'attack_vs_defence_a_venue',
    'league_log_baseline_home', 'league_log_baseline_away',
    'prior_match_count_home', 'prior_match_count_away',
    'home_cf_ewm_hl5', 'home_ca_ewm_hl5', 'away_cf_ewm_hl5', 'away_ca_ewm_hl5',
    'home_gf_l20', 'home_ga_l20', 'home_gd_l20',
    'away_gf_l20', 'away_ga_l20', 'away_gd_l20',
    'home_cf_std_l20', 'home_ca_std_l20', 'away_cf_std_l20', 'away_ca_std_l20',
    'form_resid_h', 'form_resid_a', 'gd_diff',
    'p_h_1x2', 'p_a_1x2', 'p_d_1x2',
]
target_cols = ['home_corners', 'away_corners']
categorical_cols = ['competition_id', 'season_id']

cutoff = features['date_time'].quantile(0.8)
train_feat = features[features['date_time'] <  cutoff].copy()
val_feat   = features[features['date_time'] >= cutoff].copy()

all_cat = pd.concat([train_feat[categorical_cols], val_feat[categorical_cols],
                     betting_match_features[categorical_cols]], axis=0).astype('category')
cat_dtype = {c: all_cat[c].dtype for c in categorical_cols}
for df in [train_feat, val_feat, betting_match_features]:
    for c in categorical_cols:
        df[c] = df[c].astype(cat_dtype[c])

print(pd.DataFrame({
    'set':       ['train', 'val', 'betting (holdout)'],
    'n_matches': [len(train_feat), len(val_feat), len(betting_match_features)],
    'date_min':  [train_feat['date_time'].min().date(), val_feat['date_time'].min().date(),
                  betting_match_features['date_time'].min().date()],
    'date_max':  [train_feat['date_time'].max().date(), val_feat['date_time'].max().date(),
                  betting_match_features['date_time'].max().date()],
}).to_string(index=False))
print(f'\n{len(feature_cols)} features | val cutoff: {cutoff.date()}')

              set  n_matches   date_min   date_max
            train       8956 2020-08-08 2023-10-21
              val       2242 2023-10-21 2024-12-08
betting (holdout)       1911 2024-08-09 2025-05-11

43 features | val cutoff: 2023-10-21


## Train models: LightGBM + XGBoost on home / away / diff

**Why L2 (squared error), not MAE or Poisson.**
- **MAE** (`regression_l1`) optimum is the conditional median, biased low for
  right-skewed counts.
- **Poisson** with log link squashes predictions toward the mean via `exp()`
  on bounded tree outputs.
- **L2** optimises the conditional mean directly in raw scale.

**Diff model encourages |home - away| spread.** A direct `home`/`away`
blend tends to compress the predicted diff toward 0 because the GBT
averages over many leaves and pulls every prediction toward the league
mean (e.g. on match 12449945 with 1666 vs 1649, Stage 1a's log-lambda
gave home=2.88, away=6.07 -- a -3.18 expected diff -- but the GBT
predicted 6.27 / 4.56, +1.71 diff, inverted the sign). Adding a third
L2 model on `corner_diff = home - away` gives the trees a target where
the *only* signal that matters is matchup asymmetry: Stage 1a's
`strength_diff`, `attack_vs_defence_*`, the goal-difference rolling,
and 1X2 priors. The diff model will split hard on big-strength-diff
matchups without averaging them away.

**Final blend** averages two estimators of each side: the direct
prediction and the indirect estimate via the other side + diff:

```
mu_h_final = (mu_h + (mu_a + mu_d)) / 2
mu_a_final = (mu_a + (mu_h - mu_d)) / 2
```

So `mu_h_final - mu_a_final = ((mu_h - mu_a) + mu_d) / 2` -- half
weight on the direct difference, half on the dedicated diff model.

**Tightened regularisation.** LGB: `num_leaves=63`,
`min_child_samples=20`, `reg_lambda=1.0`. XGB: `max_depth=8`,
`min_child_weight=10`, `reg_lambda=1.0`.

In [11]:
X_train, X_val = train_feat[feature_cols], val_feat[feature_cols]
X_bet = betting_match_features[feature_cols]
y_train_h, y_train_a = train_feat['home_corners'], train_feat['away_corners']
y_val_h,   y_val_a   = val_feat['home_corners'],   val_feat['away_corners']
y_train_d = y_train_h - y_train_a
y_val_d   = y_val_h - y_val_a
sw = train_feat['iw'].fillna(1) if 'iw' in train_feat.columns else None

lgb_kw = dict(objective='regression', metric='rmse',
              n_estimators=4000, learning_rate=0.02,
              num_leaves=63, min_child_samples=20,
              reg_alpha=0.0, reg_lambda=1.0,
              subsample=0.9, subsample_freq=1, colsample_bytree=0.9,
              random_state=42, verbosity=-1)
xgb_kw = dict(objective='reg:squarederror', eval_metric='rmse',
              n_estimators=4000, learning_rate=0.02,
              max_depth=8, min_child_weight=10,
              gamma=0.0, reg_alpha=0.0, reg_lambda=1.0,
              subsample=0.9, colsample_bytree=0.9,
              enable_categorical=True, tree_method='hist',
              early_stopping_rounds=150, random_state=42, n_jobs=2)

def fit_lgb(y_tr, y_va):
    m = LGBMRegressor(**lgb_kw)
    m.fit(X_train, y_tr, sample_weight=sw,
          eval_set=[(X_val, y_va)], callbacks=[early_stopping(150, verbose=False)],
          categorical_feature=categorical_cols)
    return m

def fit_xgb(y_tr, y_va):
    m = XGBRegressor(**xgb_kw)
    m.fit(X_train, y_tr, sample_weight=sw, eval_set=[(X_val, y_va)], verbose=False)
    return m

lgb_h, lgb_a, lgb_d = fit_lgb(y_train_h, y_val_h), fit_lgb(y_train_a, y_val_a), fit_lgb(y_train_d, y_val_d)
xgb_h, xgb_a, xgb_d = fit_xgb(y_train_h, y_val_h), fit_xgb(y_train_a, y_val_a), fit_xgb(y_train_d, y_val_d)

mu_h_lgb = np.maximum(lgb_h.predict(X_val), 0.1)
mu_a_lgb = np.maximum(lgb_a.predict(X_val), 0.1)
mu_d_lgb = lgb_d.predict(X_val)
mu_h_xgb = np.maximum(xgb_h.predict(X_val), 0.1)
mu_a_xgb = np.maximum(xgb_a.predict(X_val), 0.1)
mu_d_xgb = xgb_d.predict(X_val)

mu_h_direct = 0.5 * mu_h_lgb + 0.5 * mu_h_xgb
mu_a_direct = 0.5 * mu_a_lgb + 0.5 * mu_a_xgb
mu_d        = 0.5 * mu_d_lgb + 0.5 * mu_d_xgb

mu_h = 0.5 * (mu_h_direct + (mu_a_direct + mu_d))
mu_a = 0.5 * (mu_a_direct + (mu_h_direct - mu_d))
mu_h = np.maximum(mu_h, 0.1)
mu_a = np.maximum(mu_a, 0.1)

mu_h_train_direct = 0.5 * np.maximum(lgb_h.predict(X_train), 0.1) + 0.5 * np.maximum(xgb_h.predict(X_train), 0.1)
mu_a_train_direct = 0.5 * np.maximum(lgb_a.predict(X_train), 0.1) + 0.5 * np.maximum(xgb_a.predict(X_train), 0.1)
mu_d_train        = 0.5 * lgb_d.predict(X_train) + 0.5 * xgb_d.predict(X_train)
mu_h_train = 0.5 * (mu_h_train_direct + (mu_a_train_direct + mu_d_train))
mu_a_train = 0.5 * (mu_a_train_direct + (mu_h_train_direct - mu_d_train))

print(f'lgb best_iter  home={lgb_h.best_iteration_}  away={lgb_a.best_iteration_}  diff={lgb_d.best_iteration_}')
print(f'xgb best_iter  home={xgb_h.best_iteration}  away={xgb_a.best_iteration}  diff={xgb_d.best_iteration}')
print(f'pred std  home={np.std(mu_h):.2f}  away={np.std(mu_a):.2f}  diff={np.std(mu_h - mu_a):.2f}  (direct h-a std={np.std(mu_h_direct - mu_a_direct):.2f}  diff-model std={np.std(mu_d):.2f})')
print(f'pred range home=[{mu_h.min():.2f},{mu_h.max():.2f}]  away=[{mu_a.min():.2f},{mu_a.max():.2f}]  diff=[{(mu_h-mu_a).min():.2f},{(mu_h-mu_a).max():.2f}]')
print(f'train_MAE home={mean_absolute_error(y_train_h, mu_h_train):.4f}  away={mean_absolute_error(y_train_a, mu_a_train):.4f}')
print(f'val_MAE   home={mean_absolute_error(y_val_h, mu_h):.4f}  away={mean_absolute_error(y_val_a, mu_a):.4f}  diff={mean_absolute_error(y_val_d, mu_h - mu_a):.4f}')

lgb best_iter  home=184  away=138  diff=135
xgb best_iter  home=142  away=87  diff=160
pred std  home=0.88  away=0.66  diff=1.43  (direct h-a std=1.41  diff-model std=1.43)
pred range home=[2.98,8.56]  away=[2.60,7.19]  diff=[-4.17,5.57]
train_MAE home=1.7197  away=1.6285
val_MAE   home=2.2777  away=1.9944  diff=3.3513


## Evaluate on validation

MAE is the headline (corner-count error in absolute terms). RMSE penalises large
misses more heavily. NLL and CRPS treat the predicted μ as a Poisson rate and
are more relevant for downstream Q2 EV (which needs probability mass on each
corner count, not just the mean).

In [12]:
def poisson_nll(y, mu):
    mu = np.clip(mu, 1e-6, None)
    return -poisson.logpmf(np.asarray(y, int), mu).mean()

def poisson_crps(y, mu, k_max=40):
    y, mu = np.asarray(y, int), np.asarray(mu, float)
    ks = np.arange(k_max).reshape(-1, 1)
    F = poisson.cdf(ks, mu.reshape(1, -1))
    I = (ks >= y.reshape(1, -1)).astype(float)
    return np.mean(np.sum((F - I) ** 2, axis=0))

def evaluate(name, mu_h, mu_a):
    mu_d = mu_h - mu_a
    return {
        'model':     name,
        'home_MAE':  mean_absolute_error(y_val_h, mu_h),
        'away_MAE':  mean_absolute_error(y_val_a, mu_a),
        'diff_MAE':  mean_absolute_error(y_val_d, mu_d),
        'home_RMSE': np.sqrt(mean_squared_error(y_val_h, mu_h)),
        'away_RMSE': np.sqrt(mean_squared_error(y_val_a, mu_a)),
        'diff_RMSE': np.sqrt(mean_squared_error(y_val_d, mu_d)),
        'home_NLL':  poisson_nll(y_val_h, mu_h),
        'away_NLL':  poisson_nll(y_val_a, mu_a),
        'home_CRPS': poisson_crps(y_val_h, mu_h),
        'away_CRPS': poisson_crps(y_val_a, mu_a),
    }

mean_h = np.full(len(val_feat), y_train_h.mean())
mean_a = np.full(len(val_feat), y_train_a.mean())

results = pd.DataFrame([
    evaluate('Mean baseline', mean_h,   mean_a),
    evaluate('Direct LGB',    mu_h_lgb, mu_a_lgb),
    evaluate('Direct XGB',    mu_h_xgb, mu_a_xgb),
    evaluate('Direct blend',  mu_h_direct, mu_a_direct),
    evaluate('Diff-blend',    mu_h, mu_a),
]).sort_values('home_MAE').reset_index(drop=True)

results.to_csv('q1_model_comparison.csv', index=False)
display(results.round(4))

,model,home_MAE,away_MAE,diff_MAE,home_RMSE,away_RMSE,diff_RMSE,home_NLL,away_NLL,home_CRPS,away_CRPS
0,Diff-blend,2.2777,1.9944,3.3513,2.8875,2.5178,4.2231,2.4812,2.3160,1.6146,1.3971
1,Direct blend,2.2779,1.9914,3.3471,2.8887,2.5148,4.2212,2.4815,2.3145,1.6155,1.3939
2,Direct LGB,2.2798,1.9814,3.3380,2.8879,2.5065,4.2059,2.4812,2.3094,1.6158,1.3885
3,Direct XGB,2.2860,2.0087,3.3689,2.8992,2.5316,4.2511,2.4869,2.3241,1.6212,1.4045
4,Mean baseline,2.3906,2.0909,3.5652,3.0446,2.6311,4.5351,2.5631,2.3816,1.7080,1.4640


## Distance to Poisson irreducible floor

`captured_pct` = how much of the *improvable gap* (mean_baseline - floor) the
model captures. Anything in 15-30% range is realistic given only corner +
goal history; the remaining gap to the floor is structural noise we cannot
beat without additional data.

In [13]:
floor = bound.set_index('target')['simulated_MAE']
mb = results[results['model'] == 'Mean baseline'].iloc[0]
best = results[results['model'] == 'Diff-blend'].iloc[0]
gap = pd.DataFrame({
    'mean_baseline_MAE':    [mb['home_MAE'], mb['away_MAE']],
    'irreducible_floor':    [floor['home_corners'], floor['away_corners']],
    f"{best['model']}_MAE": [best['home_MAE'], best['away_MAE']],
}, index=['home', 'away'])
gap['improvable_gap'] = gap['mean_baseline_MAE'] - gap['irreducible_floor']
gap['captured_pct']   = 100 * (gap['mean_baseline_MAE'] - gap[f"{best['model']}_MAE"]) / gap['improvable_gap']
display(gap.round(3))

,mean_baseline_MAE,irreducible_floor,Diff-blend_MAE,improvable_gap,captured_pct
home,2.391,1.864,2.278,0.526,21.459
away,2.091,1.696,1.994,0.395,24.457


## Stage 3: dispersion head + A/B partition for clean Q2 holdout

Split betting matches **chronologically** into two partitions:

- **A** — first 40% (~750 matches, ~Aug–Nov 2024) — calibration set
- **B** — remaining 60% (~1100 matches, ~Dec 2024 – May 2025) — Q2 holdout

The dispersion head fits on partition A only:

```
log σ²_h  ~  α + β·μ_h + γ·|p_h_1x2 − 0.5| + δ·home_cf_std_l20
```

A simple OLS in log-variance space, with the residuals being
`log((y − μ)² + ε)` so it's well-defined even when y ≈ μ. This produces
**only a variance** for each match; **μ stays untouched** — Q2's EV calc
still finds disagreement between model μ and the OU/HC line.

Why fit in log-space: variance is positive and right-skewed, so a
linear model on the raw scale is biased; log-OLS gives an unbiased
multiplicative dispersion that NB consumes naturally.

Calibration audit on A: compare predicted σ² to realised |y − μ|² in
quantile buckets. On B, only the metrics that don't peek at outcomes
(spread of σ², range of μ) — the EV results live in Q2.

In [14]:
from team_variance import fit_dispersion

print('=== Predicted std vs theoretical ceiling (val) ===')
for name, mu, y in [('home', mu_h, y_val_h), ('away', mu_a, y_val_a)]:
    lam = float(y.mean())
    ceiling = float(np.sqrt(max(y.var() - lam, 0.01)))
    pred_std = float(np.std(mu))
    print(f'{name:6s}  actual_std={float(y.std()):.2f}  ceiling={ceiling:.2f}  pred_std={pred_std:.2f}  captured={100*pred_std/ceiling:.0f}%')

print('\n=== Diff calibration on val ===')
diff_actual = (y_val_h - y_val_a).values
diff_pred   = mu_h - mu_a
print(f'diff actual std={diff_actual.std():.2f}  diff pred std={diff_pred.std():.2f}  ratio={diff_pred.std()/diff_actual.std():.2f}')
print(f'diff actual range=[{diff_actual.min()},{diff_actual.max()}]  pred range=[{diff_pred.min():.1f},{diff_pred.max():.1f}]')

print('\n=== Stage 3: split betting into A (calibration) / B (Q2 holdout) ===')
bet_sorted = betting_match_features.sort_values('date_time').reset_index(drop=True)
n_a = int(len(bet_sorted) * 0.40)
bet_sorted['partition'] = ['A'] * n_a + ['B'] * (len(bet_sorted) - n_a)
betting_match_features = bet_sorted

X_bet = betting_match_features[feature_cols]
mu_h_bet_direct = 0.5 * np.maximum(lgb_h.predict(X_bet), 0.1) + 0.5 * np.maximum(xgb_h.predict(X_bet), 0.1)
mu_a_bet_direct = 0.5 * np.maximum(lgb_a.predict(X_bet), 0.1) + 0.5 * np.maximum(xgb_a.predict(X_bet), 0.1)
mu_d_bet        = 0.5 * lgb_d.predict(X_bet) + 0.5 * xgb_d.predict(X_bet)
mu_h_bet = np.maximum(0.5 * (mu_h_bet_direct + (mu_a_bet_direct + mu_d_bet)), 0.1)
mu_a_bet = np.maximum(0.5 * (mu_a_bet_direct + (mu_h_bet_direct - mu_d_bet)), 0.1)

# Fit dispersion on partition A only.
A_mask = (betting_match_features['partition'] == 'A').values
y_h_A = betting_match_features.loc[A_mask, 'home_corners'].astype(float).values
y_a_A = betting_match_features.loc[A_mask, 'away_corners'].astype(float).values
mu_h_A, mu_a_A = mu_h_bet[A_mask], mu_a_bet[A_mask]
ph_A = betting_match_features.loc[A_mask, 'p_h_1x2'].fillna(1/3).values
pa_A = betting_match_features.loc[A_mask, 'p_a_1x2'].fillna(1/3).values
std_h_A = betting_match_features.loc[A_mask, 'home_cf_std_l20'].fillna(global_std_c).values
std_a_A = betting_match_features.loc[A_mask, 'away_cf_std_l20'].fillna(global_std_c).values
market_certainty_h_A = np.abs(ph_A - 0.5)
market_certainty_a_A = np.abs(pa_A - 0.5)

dispersion_home = fit_dispersion(mu_h_A, y_h_A, market_certainty_h_A, std_h_A)
dispersion_away = fit_dispersion(mu_a_A, y_a_A, market_certainty_a_A, std_a_A)

print(f'partition A: n={A_mask.sum()}  date_max={betting_match_features.loc[A_mask, "date_time"].max().date()}')
print(f'partition B: n={(~A_mask).sum()}  date_min={betting_match_features.loc[~A_mask, "date_time"].min().date()}')
print(f'dispersion_home  intercept={dispersion_home.intercept:.3f}  '
      f'beta(mu)={dispersion_home.coefficients[0]:.3f}  '
      f'beta(market_cert)={dispersion_home.coefficients[1]:.3f}  '
      f'beta(rolling_std)={dispersion_home.coefficients[2]:.3f}  '
      f'calibration_scale={dispersion_home.calibration_scale:.3f}')
print(f'dispersion_away  intercept={dispersion_away.intercept:.3f}  '
      f'beta(mu)={dispersion_away.coefficients[0]:.3f}  '
      f'beta(market_cert)={dispersion_away.coefficients[1]:.3f}  '
      f'beta(rolling_std)={dispersion_away.coefficients[2]:.3f}  '
      f'calibration_scale={dispersion_away.calibration_scale:.3f}')

# Predict for all betting matches.
ph_all = betting_match_features['p_h_1x2'].fillna(1/3).values
pa_all = betting_match_features['p_a_1x2'].fillna(1/3).values
std_h_all = betting_match_features['home_cf_std_l20'].fillna(global_std_c).values
std_a_all = betting_match_features['away_cf_std_l20'].fillna(global_std_c).values
sigma2_h_bet = dispersion_home.predict(mu_h_bet, np.abs(ph_all - 0.5), std_h_all)
sigma2_a_bet = dispersion_away.predict(mu_a_bet, np.abs(pa_all - 0.5), std_a_all)

print('\n=== Dispersion calibration audit on partition A (home) ===')
sigma2_h_A_bc = dispersion_home.predict(mu_h_A, market_certainty_h_A, std_h_A)
audit = pd.DataFrame({
    'mu_bucket': pd.qcut(mu_h_A, 5, duplicates='drop'),
    'realised_sq_resid': (y_h_A - mu_h_A) ** 2,
    'pred_sigma2': sigma2_h_A_bc,
})
audit_g = audit.groupby('mu_bucket', observed=True).agg(
    n=('realised_sq_resid', 'size'),
    realised_var=('realised_sq_resid', 'mean'),
    pred_var=('pred_sigma2', 'mean'),
)
audit_g['ratio'] = audit_g['realised_var'] / audit_g['pred_var']
display(audit_g.round(3))

print('\n=== Edge preservation: (mu_h + mu_a) vs OU line on partition B ===')
B = betting_match_features[~A_mask].copy()
mu_t_B = (mu_h_bet + mu_a_bet)[~A_mask]
ou_B = (betting[betting['odds_type'] == 'OU'].drop_duplicates('match_id')
        [['match_id', 'od']].rename(columns={'od': 'ou_line'}))
B = B.merge(ou_B, on='match_id', how='left')
diff_t = (mu_t_B - B['ou_line']).dropna()
print(f'B with OU line: n={len(diff_t)}  median |mu_t - OU_line|={float(diff_t.abs().median()):.3f}')

print('\n=== Diff sanity: spot-check match 12449945 ===')
mid_row = betting_match_features[betting_match_features.match_id==12449945]
if len(mid_row):
    pos = list(betting_match_features.match_id).index(12449945)
    print(f'  pred_home={mu_h_bet[pos]:.2f}  pred_away={mu_a_bet[pos]:.2f}  pred_diff={mu_h_bet[pos]-mu_a_bet[pos]:+.2f}  '
          f'(direct h-a={mu_h_bet_direct[pos]-mu_a_bet_direct[pos]:+.2f}, diff_model={mu_d_bet[pos]:+.2f})')
    print(f'  actual: home={int(mid_row.iloc[0].home_corners)}  away={int(mid_row.iloc[0].away_corners)}')
    print(f'  sigma2_home={sigma2_h_bet[pos]:.2f}  sigma2_away={sigma2_a_bet[pos]:.2f}')

=== Predicted std vs theoretical ceiling (val) ===
home    actual_std=3.03  ceiling=1.87  pred_std=0.88  captured=47%
away    actual_std=2.63  ceiling=1.59  pred_std=0.66  captured=42%

=== Diff calibration on val ===
diff actual std=4.52  diff pred std=1.43  ratio=0.32
diff actual range=[-13,18]  pred range=[-4.2,5.6]

=== Stage 3: split betting into A (calibration) / B (Q2 holdout) ===
partition A: n=764  date_max=2024-12-14
partition B: n=1147  date_min=2024-12-14
dispersion_home  intercept=0.405  beta(mu)=0.181  beta(market_cert)=-0.213  beta(rolling_std)=0.003  calibration_scale=2.065
dispersion_away  intercept=0.451  beta(mu)=0.114  beta(market_cert)=-0.694  beta(rolling_std)=0.136  calibration_scale=2.006

=== Dispersion calibration audit on partition A (home) ===


,n,realised_var,pred_var,ratio
mu_bucket,,,,
"(3.253, 5.157]",153,5.368,7.022,0.765
"(5.157, 5.588]",153,8.070,8.116,0.994
"(5.588, 5.98]",152,10.371,8.747,1.186
"(5.98, 6.443]",153,9.626,9.395,1.025
"(6.443, 8.944]",153,10.564,10.709,0.986



=== Edge preservation: (mu_h + mu_a) vs OU line on partition B ===
B with OU line: n=603  median |mu_t - OU_line|=0.726

=== Diff sanity: spot-check match 12449945 ===
  pred_home=5.54  pred_away=4.44  pred_diff=+1.10  (direct h-a=+1.33, diff_model=+1.10)
  actual: home=15  away=1
  sigma2_home=8.31  sigma2_away=6.76


## Save predictions for Q2

Validation predictions document model performance. Betting predictions
feed Q2 with: mu_h, mu_a (from the diff-blend), sigma2_h, sigma2_a
(from the dispersion head fit on partition A), and the partition
column. Q2 reads `partition == 'B'` to pick the clean Q2 holdout
subset for EV evaluation.

No total-corner prediction is saved; Q2 should derive total directly
from the Negative Binomial convolution of (mu_h, sigma2_h) and
(mu_a, sigma2_a).

In [15]:
val_keep = ['match_id', 'date_time', 'competition_id', 'season_id',
            'home_corners', 'away_corners']
val_pred = val_feat[val_keep].copy()
val_pred['pred_home_lgb_direct']  = mu_h_lgb
val_pred['pred_away_lgb_direct']  = mu_a_lgb
val_pred['pred_diff_lgb']         = mu_d_lgb
val_pred['pred_home_xgb_direct']  = mu_h_xgb
val_pred['pred_away_xgb_direct']  = mu_a_xgb
val_pred['pred_diff_xgb']         = mu_d_xgb
val_pred['pred_home_direct']      = mu_h_direct
val_pred['pred_away_direct']      = mu_a_direct
val_pred['pred_diff']             = mu_d
val_pred['pred_home']             = mu_h
val_pred['pred_away']             = mu_a
val_pred.to_csv('q1_validation_predictions.csv', index=False)

bet_pred = betting_match_features[['match_id', 'date_time', 'competition_id', 'season_id',
                                   'home_corners', 'away_corners', 'partition']].copy()
bet_pred['pred_home_corners']     = mu_h_bet
bet_pred['pred_away_corners']     = mu_a_bet
bet_pred['pred_corner_diff']      = mu_h_bet - mu_a_bet
bet_pred['sigma2_home']           = sigma2_h_bet
bet_pred['sigma2_away']           = sigma2_a_bet
bet_pred.to_csv('q1_betting_match_predictions.csv', index=False)

print('Saved q1_validation_predictions.csv and q1_betting_match_predictions.csv')
print(f'Betting partition counts: A={(betting_match_features.partition=="A").sum()}  '
      f'B={(betting_match_features.partition=="B").sum()}')
print(f'pred_corner_diff: std={(mu_h_bet-mu_a_bet).std():.3f}  range=[{(mu_h_bet-mu_a_bet).min():.2f},{(mu_h_bet-mu_a_bet).max():.2f}]')
print(f'sigma2_home: median={np.median(sigma2_h_bet):.2f}  sigma2_away: median={np.median(sigma2_a_bet):.2f}')

Saved q1_validation_predictions.csv and q1_betting_match_predictions.csv
Betting partition counts: A=764  B=1147
pred_corner_diff: std=1.512  range=[-3.55,5.84]
sigma2_home: median=8.74  sigma2_away: median=6.87


## Q1 summary

- **Stage 1a -- sequential Bayesian Poisson team-strength model.**
  Walks ~13k matches chronologically with Elo-style updates of (att,
  def) latents in log-corner space; outputs leak-free per-match
  strength features (general + venue-split).
- **Stage 1b -- recent form + goals + variance.** Short EWM (recent
  form), 20-game goal rolling, 20-game corner std.
- **Stage 2 -- LightGBM + XGBoost blend** on 43 features. Three
  targets: home, away, **diff** (home - away). Final blend averages
  direct estimates with the indirect estimate via the other side +
  diff: forces predictions apart on asymmetric matchups.
- **1X2 implied probabilities** as features for the betting set only.
- **Stage 3 -- dispersion head + clean Q2 holdout.** Betting matches
  split chronologically into A (40%) and B (60%). Dispersion fit on A;
  mu untouched.
- **Output for Q2**: `q1_betting_match_predictions.csv` with
  (match_id, partition, pred_home/away_corners, pred_corner_diff,
  sigma2_home, sigma2_away).